# CatalogAugment AI: Product Image Augmentation Pipeline
**End-to-End Reproducible Notebook (Weeks 1–4 Deliverables)**

- **Domain / Industry**: Retail & E-Commerce
- **Use Case**: Product image augmentation pipeline for visual-search & classification robustness
- **Core Stack**: `Albumentations`, Hugging Face `Diffusers`, `Bria RMBG-1.4`, OpenCV 3D Homography, `PyTorch` (`ResNet-18`), `scikit-learn`

This notebook orchestrates and visualizes all 4 milestones:
1. **Week 1**: Multi-Category Product Dataset Preparation, Preprocessing & Baseline EDA
2. **Week 2**: Conventional Augmentation Pipeline (`Albumentations`) + Realistic vs. Unrealistic Quality Audit
3. **Week 3**: Controlled Generative Augmentation (`Diffusers` + RMBG-1.4 + 3D Angles + Quality Gate) & PyTorch Data Pipeline
4. **Week 4**: Controlled PyTorch Robustness Evaluation (Baseline vs. Albumentations vs. Diffusers vs. Combined) & Visual Analysis

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image as IPyImage

BASE_DIR = Path('.').resolve()
OUT_DIR = BASE_DIR / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
print('Pipeline directory:', BASE_DIR)

## Week 1: Product Dataset Preparation & Baseline EDA
Streams 240 real high-resolution e-commerce studio images across 6 retail categories (`Shirts`, `Casual Shoes`, `Watches`, `Backpacks`, `Sunglasses`, `Handbags`), validates image hygiene, pads/normalizes to `512x512` RGB, and builds stratified `train (70%) / val (15%) / test (15%)` splits.

In [ ]:
# Run Week 1 script if outputs are not yet generated
if not (OUT_DIR / 'week1_dataset_stats.json').exists():
    %run 01_dataset_preparation.py

with open(OUT_DIR / 'week1_dataset_stats.json') as f:
    w1_stats = json.load(f)
print(json.dumps(w1_stats, indent=2))

display(IPyImage(filename=str(FIG_DIR / 'week1_class_distribution.png')))
display(IPyImage(filename=str(FIG_DIR / 'week1_eda_stats.png')))
display(IPyImage(filename=str(FIG_DIR / 'week1_sample_grid.png')))

## Week 2: Conventional Product Image Augmentation (`Albumentations`)
Applies domain-calibrated `Albumentations` transformations (orientation, 3D perspective, lighting/gamma, motion/Gaussian blur, ISO noise, JPEG compression, mild occlusion) strictly to the `train` split (`168 x 3 = 504` variants), plus an explicit **Realistic vs. Unrealistic** quality audit.

In [ ]:
if not (OUT_DIR / 'week2_albumentations_manifest.csv').exists():
    %run 02_albumentations_pipeline.py

with open(OUT_DIR / 'week2_albumentations_config.json') as f:
    w2_cfg = json.load(f)
display(pd.DataFrame(w2_cfg['rationale']))

display(IPyImage(filename=str(FIG_DIR / 'week2_albumentations_comparison.png')))
display(IPyImage(filename=str(FIG_DIR / 'week2_realistic_vs_unrealistic.png')))

## Week 3 (Part A): Controlled Generative Augmentation (`Diffusers` + RMBG-1.4 + 3D Angles)
Segments every training product with **Bria RMBG-1.4**, applies 3D perspective homography warps (yaw/pitch/roll), inpaints 6 realistic customer-photo environments via Hugging Face **`diffusers.StableDiffusionInpaintPipeline`**, composites the identity-locked product foreground, and validates every sample through an automated quality gate.

In [ ]:
if not (OUT_DIR / 'week3_quality_report.json').exists():
    %run 03_diffusers_generative.py

with open(OUT_DIR / 'week3_quality_report.json') as f:
    w3_rep = json.load(f)
print(json.dumps(w3_rep, indent=2))

display(IPyImage(filename=str(FIG_DIR / 'week3_diffusers_showcase_grid.png')))
display(IPyImage(filename=str(FIG_DIR / 'week3_quality_audit_pass_vs_reject.png')))

## Week 3 (Part B) & Week 4: PyTorch Integration & Robustness Evaluation
Trains 4 identical PyTorch `ResNet-18` models (Classification + 256-dim L2-normalized Visual-Search Embedding Head) across:
- **Exp A**: Baseline (Clean studio only)
- **Exp B**: Conventional (`Clean + Albumentations`)
- **Exp C**: Generative (`Clean + Diffusers`)
- **Exp D**: Combined / CatalogAugment AI (`Clean + Albumentations + Diffusers`)

Evaluates all 4 models on an independent, held-out 5-condition real-world stress test suite (`Clean`, `Lighting Shift`, `3D Angle Shift`, `Unseen Cluttered Background`, and `Combined Hard Customer Smartphone Photo`).

In [ ]:
if not (OUT_DIR / 'week4_evaluation_table.csv').exists():
    %run 04_pytorch_train_and_evaluate.py

eval_df = pd.read_csv(OUT_DIR / 'week4_evaluation_table.csv')
display(eval_df.pivot(index='model', columns='condition', values='accuracy').round(4) * 100)
display(eval_df.pivot(index='model', columns='condition', values='recall_at_1').round(4) * 100)

display(IPyImage(filename=str(FIG_DIR / 'week4_test_stress_conditions.png')))
display(IPyImage(filename=str(FIG_DIR / 'week4_training_curves.png')))
display(IPyImage(filename=str(FIG_DIR / 'week4_robustness_comparison.png')))
display(IPyImage(filename=str(FIG_DIR / 'week4_confusion_matrices.png')))
display(IPyImage(filename=str(FIG_DIR / 'week4_tsne_embeddings.png')))
display(IPyImage(filename=str(FIG_DIR / 'week4_retrieval_examples.png')))